# Funding Calls RAG with ChromaDB

This notebook prepares a retrieval dataset from local funding call PDFs. The output is a persistent ChromaDB collection that can be used in the next step for question answering.

## Setup

Install required packages in the active environment.

In [ ]:
%pip install -q chromadb pypdf sentence-transformers tqdm

## Imports

In [ ]:
from pathlib import Path
import hashlib
import re
from typing import Dict, List

from pypdf import PdfReader
from tqdm.auto import tqdm

import chromadb
from chromadb.utils import embedding_functions

## Configuration

Paths and chunking settings are centralized here for easy reruns.

In [ ]:
PROJECT_ROOT = Path.cwd()
PDF_DIR = PROJECT_ROOT / "docs" / "fundingcalls"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
COLLECTION_NAME = "funding_calls"

CHUNK_SIZE = 1800
CHUNK_OVERLAP = 250
MIN_CHUNK_CHARS = 120

RESET_COLLECTION = True
EXCLUDE_FILES = set()

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF directory: {PDF_DIR}")
print(f"Chroma directory: {CHROMA_DIR}")

## Source Files

List the PDF files that will be ingested.

In [ ]:
pdf_paths = sorted([p for p in PDF_DIR.glob("*.pdf") if p.name not in EXCLUDE_FILES])

print(f"Found {len(pdf_paths)} PDF files")
for path in pdf_paths:
    print(" -", path.name)

if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in {PDF_DIR}")

## Text Extraction

Text is extracted page by page and lightly cleaned before chunking.

In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_pages_from_pdf(pdf_path: Path) -> List[Dict]:
    rows: List[Dict] = []
    reader = PdfReader(str(pdf_path))

    for page_idx, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        text = clean_text(raw_text)

        if len(text) < 40:
            continue

        rows.append(
            {
                "source_file": pdf_path.name,
                "source_path": str(pdf_path),
                "page": page_idx,
                "text": text,
            }
        )

    return rows

In [ ]:
page_rows: List[Dict] = []
for pdf_path in tqdm(pdf_paths, desc="Extracting PDF pages"):
    page_rows.extend(extract_pages_from_pdf(pdf_path))

print(f"Extracted {len(page_rows)} non-empty pages")
page_rows[0] if page_rows else "No text extracted"

## Chunking

Overlapping chunks are created to improve retrieval around text boundaries.

In [ ]:
def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if overlap >= size:
        raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_SIZE")

    chunks: List[str] = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + size, text_len)
        chunk = text[start:end].strip()

        if len(chunk) >= MIN_CHUNK_CHARS:
            chunks.append(chunk)

        if end == text_len:
            break

        start = end - overlap

    return chunks

In [ ]:
documents: List[str] = []
metadatas: List[Dict] = []
ids: List[str] = []

for row in tqdm(page_rows, desc="Chunking pages"):
    chunks = chunk_text(row["text"])

    for chunk_idx, chunk in enumerate(chunks):
        raw_id = f"{row['source_file']}|{row['page']}|{chunk_idx}|{len(chunk)}"
        chunk_id = hashlib.md5(raw_id.encode("utf-8")).hexdigest()

        documents.append(chunk)
        metadatas.append(
            {
                "source_file": row["source_file"],
                "source_path": row["source_path"],
                "page": row["page"],
                "chunk_index": chunk_idx,
                "chunk_chars": len(chunk),
            }
        )
        ids.append(chunk_id)

print(f"Created {len(documents)} chunks")
if documents:
    print("Sample chunk preview:")
    print(documents[0][:500])

## ChromaDB Indexing

Chunks are embedded with a multilingual model and written to a persistent collection.

In [ ]:
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-mpnet-base-v2"
)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

if RESET_COLLECTION:
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embed_fn,
    metadata={"description": "Funding call chunks for RAG retrieval"},
)

BATCH_SIZE = 128
for i in tqdm(range(0, len(documents), BATCH_SIZE), desc="Upserting to Chroma"):
    collection.upsert(
        ids=ids[i:i + BATCH_SIZE],
        documents=documents[i:i + BATCH_SIZE],
        metadatas=metadatas[i:i + BATCH_SIZE],
    )

print("Collection name:", COLLECTION_NAME)
print("Stored chunks:", collection.count())

## Retrieval Check

A quick query confirms that the collection is searchable and returns grounded chunks with source metadata.

In [ ]:
query = "startup deep tech and innovation funding"
results = collection.query(query_texts=[query], n_results=3)

for i, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
    start=1,
):
    print(f"\nResult {i} | distance={dist:.4f}")
    print(meta)
    print(doc[:450], "...")